# Optimize page extractor with DSPy

In [1]:
TRAINING_MODE = "light"
EXAMPLES_PATH = "../output/dataset/reviewed.jsonl"
OPTIMIZED_PATH = f"../output/models/dspy-extract-filtered-reorder-{TRAINING_MODE}.json"
EVAL_RESULTS_PATH = OPTIMIZED_PATH.replace(".json", "-eval.jsonl")
LOG_DIR = "../output/logs"

In [3]:
import mlflow
import dspy

mlflow.dspy.autolog(
    log_compiles=True,
    log_evals=True,
    log_traces_from_compile=True
)
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("dspy-extract-optimize");

/home/mbrt/priv/PKNA/exporter/.venv/lib/python3.12/site-packages/mlflow/types/type_hints.py:234: UserWarning: Any type hint is inferred as AnyType, and MLflow doesn't validate the data for this type. Please use a more specific type hint to enable data validation.
  dtype=_infer_colspec_type_from_type_hint(effective_type).dtype,
/home/mbrt/priv/PKNA/exporter/.venv/lib/python3.12/site-packages/mlflow/types/type_hints.py:215: UserWarning: Any type hint is inferred as AnyType, and MLflow doesn't validate the data for this type. Please use a more specific type hint to enable data validation.
  dtype=Map(_infer_colspec_type_from_type_hint(type_hint=args[1]).dtype),


In [2]:
from typing import Literal
from pydantic import BaseModel, Field
import dspy


Character = Literal['uno', 'paperinik', 'other']


class ExtractedLine(BaseModel):
    """A line of dialogue extracted from a comic book page."""
    character: Character = Field(
        description="The character who said the line."
    )
    text: str = Field(
        description="The text of the dialogue line."
    )


class CharacterDescription(BaseModel):
    """Description of a character in the comic."""
    name: Character = Field(
        description="The name of the character."
    )
    description: str = Field(
        description="A description and biography of the character."
    )
    model_config = {
        "frozen": True,
    }


class DialogueExtraction(dspy.Signature):
    """Extract dialogues from a comic book page.

    The dialogues are expected to be in the form of text bubbles, typically found in comic books."""

    characters: list[CharacterDescription] = dspy.InputField(
        desc="List of character descriptions we care about."
    )
    page: dspy.Image = dspy.InputField(desc="Comic book page image")
    dialogue: list[ExtractedLine] = dspy.OutputField(
        desc="List of dialogues extracted from the page, each with a character name and text."
    )


class ReorderDialogue(dspy.Signature):
    """Reorder the dialogue lines if they appear out of order."""

    dialogue: list[ExtractedLine] = dspy.InputField(
        desc="List of dialogue lines, possibly out of order."
    )
    reordered: list[ExtractedLine] = dspy.OutputField(
        desc="List of dialogue lines reordered based on the flow of the conversation."
    )


class ExtractorModule(dspy.Module):
    """Extract dialogues from a comic book page."""

    def __init__(self, characters: list[CharacterDescription]):
        self.characters = characters
        self.extractor = dspy.ChainOfThought(DialogueExtraction)
        self.normalize = dspy.Predict(
            dspy.make_signature(
                signature='text -> normalized',
                instructions="Normalize text by using normal caps instead of all caps, remove line-break hyphens, and accented letters instead of apostrophes when appropriate.",
            )
        )
        self.reorder = dspy.ChainOfThought(ReorderDialogue)

    def forward(self, page: dspy.Image) -> dspy.Prediction:
        extracted = self.extractor(
            characters=self.characters, page=page).dialogue
        normalized = [
            self.normalize(text=line.text).normalized
            for line in extracted
        ]
        dialogue = [
            ExtractedLine(character=line.character, text=normalized_text)
            for line, normalized_text in zip(extracted, normalized)
        ]
        reordered = self.reorder(dialogue=dialogue).reordered
        return dspy.Prediction(dialogue=reordered)

In [3]:
import os
from dotenv import load_dotenv


def make_gemini_llm(name: str) -> dspy.LM:
    return dspy.LM(
        model=f'vertex_ai/{name}',
        vertex_credentials=os.getenv('VERTEX_AI_CREDS'),
        temperature=1.0,
        top_p=0.95,
        top_k=64,
        max_tokens=65535,
    )


def init_llms() -> tuple[dspy.LM, dspy.LM]:
    load_dotenv()
    task_lm = make_gemini_llm('gemini-2.5-flash')
    prompt_lm = make_gemini_llm('gemini-2.5-pro')
    dspy.configure(lm=task_lm, track_usage=True)
    return task_lm, prompt_lm


task_lm, prompt_lm = init_llms()

In [4]:
def load_character(name: Character) -> CharacterDescription:
    """Load a character description from the environment variable."""
    p = os.path.join('../input/bios', f'{name}.md')
    with open(p, 'r', encoding='utf-8') as f:
        description = f.read()
    return CharacterDescription(name=name, description=description)


def validate_precise(example: dspy.Example, pred: dspy.Prediction) -> dspy.Prediction:
    # Full score for exact match.
    if example.dialogue == pred.dialogue:
        return dspy.Prediction(score=1.0)

    # Partial scoring.
    length = 1.0 if len(example.dialogue) == len(pred.dialogue) else 0.0
    characters = 1.0 if set(
        line.character for line in example.dialogue) == set(
        line.character for line in pred.dialogue
    ) else 0.0
    # Count correct text.
    correct_text = sum(
        1 for e, p in zip(example.dialogue, pred.dialogue)
        if e.text == p.text
    ) / len(example.dialogue)
    # Correct attribution.
    attribution = sum(
        1 for e, p in zip(example.dialogue, pred.dialogue)
        if e.character == p.character
    ) / len(example.dialogue)

    # Compose feedback.
    feedback = [
        f"* Expected {len(example.dialogue)} lines, got {len(pred.dialogue)}." if length < 1.0 else None,
        f"* Expected {set(l.character for l in example.dialogue)} characters." if characters < 1.0 else None,
        f"* The text for {correct_text * len(pred.dialogue)} lines is incorrect." if correct_text < 1.0 else None,
        f"* The character attribution for {attribution * len(pred.dialogue)} lines is incorrect." if attribution< 1.0 else None,
    ]
    feedback_txt = "\n".join(f for f in feedback if f is not None)

    # Average score.
    score = (length + characters + correct_text + attribution) / 4.0
    return dspy.Prediction(score=score, feedback=feedback_txt)


def validate(example: dspy.Example, pred: dspy.Prediction, trace=None) -> float | bool:
    pred = validate_precise(example, pred)
    if trace is None:
        # We're doing optimization or evaluation.
        # Return precise score.
        return pred.score
    return pred.score >= 0.99


def validate_feedback(example: dspy.Example, pred: dspy.Prediction, trace=None) -> dspy.Prediction:
    pred = validate_precise(example, pred)
    if trace is None:
        # We're doing optimization or evaluation.
        # Return precise score.
        return pred
    return dspy.Prediction(
        score=pred.score > 0.9,
        feedback=pred.feedback,
    )


def init_model() -> ExtractorModule:
    """Initialize the extractor module with characters."""
    characters = [
        load_character('uno'),
        load_character('paperinik'),
        CharacterDescription(
            name='other',
            description='Any other character in the comic book.'
        ),
    ]
    return ExtractorModule(characters=characters)


def optimize(
        trainset: list[dspy.Example],
        valset: list[dspy.Example],
        training_mode: Literal["light", "medium"],
) -> dspy.Module:
    task_lm, prompt_lm = init_llms()
    extractor = init_model()
    teleprompter = dspy.MIPROv2(
        metric=validate,
        auto=training_mode,
        prompt_model=prompt_lm,
        task_model=task_lm,
        num_threads=8,
        verbose=True,
    )
    with mlflow.start_run() as run:
        compiled_model = teleprompter.compile(
            extractor,
            trainset=trainset,
            valset=valset,
            # requires_permission_to_run=False,
        )
        evaluation = dspy.Evaluate(devset=valset, metric=validate)
        evaluation(compiled_model)

    return compiled_model


def optimize_simba(examples: list[dspy.Example]) -> dspy.Module:
    init_llms()
    extractor = init_model()
    teleprompter = dspy.SIMBA(
        metric=validate_feedback,
        bsize=16,
        num_threads=8,
    )
    compiled_model = teleprompter.compile(extractor, trainset=examples, seed=7)
    return compiled_model

In [5]:
import json
from sklearn.model_selection import train_test_split


def classify_character(character: str) -> Character:
    """Classify the character based on the text."""
    match character.lower():
        case 'uno':
            return 'uno'
        case 'pk' | 'paperinik':
            return 'paperinik'
        case _:
            return 'other'


def parse_reviewed(line: str) -> dspy.Example | None:
    """Parse a line from the reviewed dataset."""
    data = json.loads(line)
    dialogue = data['ocr']['dialogue']

    if not dialogue or len(dialogue) == 0:
        return None

    return dspy.Example(
        path=data['image'],
        page=dspy.Image.from_file(data['image']),
        dialogue=[
            ExtractedLine(
                character=classify_character(line['character']),
                text=line['text'],
            )
            for line in dialogue
        ]
    ).with_inputs('page')


def build_dataset(examples_path: str) -> tuple[list[dspy.Example], list[dspy.Example]]:
    res = []
    with open(examples_path, 'r') as f:
        for line in f:
            example = parse_reviewed(line)
            if example:
                res.append(example)
    train, test = train_test_split(
        res, test_size=0.5, random_state=42
    )
    return train, test

In [6]:
from dspy.evaluate import Evaluate


def save_model(model: dspy.Module, output_path: str) -> None:
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    model.save(output_path)


def run_eval(model: dspy.Module, examples: list[dspy.Example]) -> None:
    evaluator = Evaluate(devset=examples, num_threads=4,
                         display_progress=True, display_table=True,
                         return_outputs=True)
    evaluator(model, metric=validate)

## Run

In [7]:
trainset, valset = build_dataset(EXAMPLES_PATH)
len(trainset), len(valset)

(10, 11)

In [12]:
if TRAINING_MODE == "simba":
    m = optimize_simba(trainset)
else:
    m = optimize(trainset, valset, TRAINING_MODE)
save_model(m, OPTIMIZED_PATH)

2025/08/09 21:56:59 INFO dspy.teleprompt.mipro_optimizer_v2: 
RUNNING WITH THE FOLLOWING LIGHT AUTO RUN SETTINGS:
num_trials: 31
minibatch: False
num_fewshot_candidates: 6
num_instruct_candidates: 3
valset size: 11



Projected Language Model (LM) Calls

Based on the parameters you have set, the maximum number of LM calls is projected as follows:

- Prompt Generation: 10 data summarizer calls + 3 * 3 lm calls in program + (4) lm calls in program-aware proposer = 23 prompt model calls
- Program Evaluation: 11 examples in val set * 31 batches = 341 LM program calls

Estimated Cost Calculation:

Total Cost = (Number of calls to task model * (Avg Input Token Length per Call * Task Model Price per Input Token + Avg Output Token Length per Call * Task Model Price per Output Token)
            + (Number of program calls * (Avg Input Token Length per Call * Task Prompt Price per Input Token + Avg Output Token Length per Call * Prompt Model Price per Output Token).

For a preliminary estimate of potential costs, we recommend you perform your own calculations based on the task
and prompt models you intend to use. If the projected costs exceed your budget or expectations, you may consider:

- Reducing the numb

2025/08/09 21:57:19 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 1: BOOTSTRAP FEWSHOT EXAMPLES <==
2025/08/09 21:57:19 INFO dspy.teleprompt.mipro_optimizer_v2: These will be used as few-shot example candidates for our program and for creating instructions.

2025/08/09 21:57:19 INFO dspy.teleprompt.mipro_optimizer_v2: Bootstrapping N=6 sets of demonstrations...



No input received within 20 seconds. Proceeding with execution...
Bootstrapping set 1/6
Bootstrapping set 2/6
Bootstrapping set 3/6


100%|██████████| 10/10 [02:51<00:00, 17.14s/it]

...


🏃 View run eval_full_31 at: http://127.0.0.1:5000/#/experiments/1/runs/57fa2ac9cf744bbb95eae0b57c4a0ca7
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


2025/08/09 22:39:31 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 55.61 with parameters ['Predictor 0: Instruction 0', 'Predictor 0: Few-Shot Set 2', 'Predictor 1: Instruction 1', 'Predictor 1: Few-Shot Set 2', 'Predictor 2: Instruction 0', 'Predictor 2: Few-Shot Set 2'].
2025/08/09 22:39:31 INFO dspy.teleprompt.mipro_optimizer_v2: Scores so far: [64.77, 56.66, 55.69, 58.41, 58.56, 63.27, 63.57, 58.42, 58.54, 57.83, 59.39, 57.35, 63.57, 64.77, 64.77, 60.99, 65.9, 62.46, 55.75, 55.61, 61.47, 58.55, 63.24, 60.32, 65.9, 55.75, 65.9, 65.9, 65.9, 56.1, 61.97, 55.61]
2025/08/09 22:39:31 INFO dspy.teleprompt.mipro_optimizer_v2: Best score so far: 65.9
2025/08/09 22:39:31 INFO dspy.teleprompt.mipro_optimizer_v2: =========================


2025/08/09 22:39:31 INFO dspy.teleprompt.mipro_optimizer_v2: Returning best identified program with score 65.9!


2025/08/09 22:39:39 INFO dspy.evaluate.evaluate: Average Metric: 7.24926948051948 / 11 (65.9%)


🏃 View run eval at: http://127.0.0.1:5000/#/experiments/1/runs/00afde1a2f5045e4ac8a5d9ffc9074f6
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run unique-fly-649 at: http://127.0.0.1:5000/#/experiments/1/runs/4c59461c23ac45a2a96fd4397a2b3ba9
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


[Trace(trace_id=tr-96876094ea45d70ceede4f7941a7a3d0), Trace(trace_id=tr-4926e908896e41cd926dc7da28ebacc2), Trace(trace_id=tr-8762625a3523e75510854a7f6a1804f3), Trace(trace_id=tr-d2cca243e57cc138639d29b79e9d930f), Trace(trace_id=tr-bdca94fc28986c90efc50136192da7e9), Trace(trace_id=tr-2a819282b4886e748f1dcc598d8e5182), Trace(trace_id=tr-bd4db6376436a83d2927f1b0790bc961), Trace(trace_id=tr-4dd7f84fd75de63f6487563840510f50), Trace(trace_id=tr-37b96d78609a882cb4adced364fbb107), Trace(trace_id=tr-b6ad9a4c0a314e5ff29fafcada70882a)]

In [ ]:
run_eval(m, valset)

Average Metric: 16.40 / 21 (78.1%): 100%|██████████| 21/21 [00:38<00:00,  1.83s/it]

2025/08/08 13:46:43 INFO dspy.evaluate.evaluate: Average Metric: 16.404676573426574 / 21 (78.1%)


,path,page,example_dialogue,pred_dialogue,validate
0,../input/pkna/pkna-18/pkna18-017.jpg,"<<CUSTOM-TYPE-START-IDENTIFIER>>[{'type': 'image_url', 'image_url'...","[ExtractedLine(character='uno', text=""Qui non c'è più.""), Extracte...","[ExtractedLine(character='uno', text=""Qui non c'è più.""), Extracte...",✔️ [0.967]
1,../input/pkna/pkna-29/pk00025.jpg,"<<CUSTOM-TYPE-START-IDENTIFIER>>[{'type': 'image_url', 'image_url'...","[ExtractedLine(character='other', text='Collegamento stabilito, Un...","[ExtractedLine(character='other', text='Collegamento stabilito, Un...",✔️ [0.864]
2,../input/pkna/pkna-30/pk00059.jpg,"<<CUSTOM-TYPE-START-IDENTIFIER>>[{'type': 'image_url', 'image_url'...","[ExtractedLine(character='other', text='Posso...'), ExtractedLine(...","[ExtractedLine(character='other', text='Posso...'), ExtractedLine(...",✔️ [0.750]
3,../input/pkna/pkna-15/pkna15-37.jpg,"<<CUSTOM-TYPE-START-IDENTIFIER>>[{'type': 'image_url', 'image_url'...","[ExtractedLine(character='uno', text='Grande colpo di scena! Nella...","[ExtractedLine(character='uno', text='Grande colpo di scena! Nella...",✔️ [0.933]
4,../input/pkna/pkna-13/buia032.jpg,"<<CUSTOM-TYPE-START-IDENTIFIER>>[{'type': 'image_url', 'image_url'...","[ExtractedLine(character='paperinik', text='Aggiornamento sulla si...","[ExtractedLine(character='paperinik', text='Aggiornamento sulla si...",✔️ [1.000]
5,../input/pkna/pkna-6/43.jpg,"<<CUSTOM-TYPE-START-IDENTIFIER>>[{'type': 'image_url', 'image_url'...","[character='uno' text='Wow! Non era poi così difficile!', characte...","[character='uno' text='Wow! Non era poi così difficile!', characte...",✔️ [1.000]
6,../input/pkna/pkna-6/15.jpg,"<<CUSTOM-TYPE-START-IDENTIFIER>>[{'type': 'image_url', 'image_url'...","[ExtractedLine(character='uno', text='allora, ecco qui il primo li...","[ExtractedLine(character='uno', text='Allora, ecco qui il primo li...",✔️ [0.750]
7,../input/pkna/pkna-1/pkna1-26.jpg,"<<CUSTOM-TYPE-START-IDENTIFIER>>[{'type': 'image_url', 'image_url'...","[ExtractedLine(character='uno', text=""E ora l'ho perso di nuovo! I...","[ExtractedLine(character='paperinik', text=""E ora l'ho perso di nu...",✔️ [0.542]
8,../input/pkna/pkna-41/pkna41-026.jpg,"<<CUSTOM-TYPE-START-IDENTIFIER>>[{'type': 'image_url', 'image_url'...","[ExtractedLine(character='uno', text=""Da quelle parti, nel 1908, s...","[ExtractedLine(character='uno', text=""Da quelle parti, nel 1908, s...",✔️ [1.000]
9,../input/pkna/pkna-4/pkna4-007.jpg,"<<CUSTOM-TYPE-START-IDENTIFIER>>[{'type': 'image_url', 'image_url'...","[ExtractedLine(character='uno', text='Ben alzato, paperino!'), Ext...","[ExtractedLine(character='uno', text='Ben alzato, Paperino!'), Ext...",✔️ [0.864]


🏃 View run eval at: http://localhost:5000/#/experiments/1/runs/4151b3c1fa97450fa5f90ae862b2f7c6
🧪 View experiment at: http://localhost:5000/#/experiments/1


[Trace(trace_id=tr-359a5496d16d5239c9763107ccce6ca5), Trace(trace_id=tr-8f4776d554b3e569ee09f1d788d83d3c), Trace(trace_id=tr-5485783e0b6bd103f9cf8c75058b8373), Trace(trace_id=tr-3b71bc1d9cfa6c71299aa8cd0ecfdcd9), Trace(trace_id=tr-b9f7ea7a12575cad37331fe3f05e15df), Trace(trace_id=tr-b2ec44284dc440e95b16a5957d7191e3), Trace(trace_id=tr-b5c621f1b92cbad184c4f59e35bf72ab), Trace(trace_id=tr-8452a9c9743775ab796b8d31be22c0a3), Trace(trace_id=tr-122f93c20211b46ffab4235a45ded5d5), Trace(trace_id=tr-c00c96138c8f7d830e81142102b80791)]

In [8]:
from tqdm.notebook import tqdm


def dialogue_to_dict(dialogue: list[ExtractedLine]) -> list[dict]:
    """Convert a dialogue to a dictionary format."""
    return [
        {'character': line.character, 'text': line.text}
        for line in dialogue
    ]


def dump_eval_results(model: dspy.Module, examples: list[dspy.Example]) -> None:
    """Dump evaluation results to a file."""
    results = []
    for example in examples:
        pred = model(example.page)
        score = validate(example, pred)
        results.append({
            'page': example.path,
            'example': dialogue_to_dict(example.dialogue),
            'prediction': dialogue_to_dict(pred.dialogue),
            'score': score,
        })
    with open(EVAL_RESULTS_PATH, 'w') as f:
        for result in tqdm(results):
            f.write(json.dumps(result, ensure_ascii=False) + '\n')

In [9]:
model = init_model()
model.load(OPTIMIZED_PATH)
dump_eval_results(model, trainset)

  0%|          | 0/10 [00:00<?, ?it/s]